# 005 · asyncio in Depth

> **Standard Library › Concurrency** · Python 3.14

The Async notebooks in the Syntax group introduced `async`/`await`, tasks and `TaskGroup` *(see Syntax › Async › 001 · Async and Await and Syntax › Async › 002 · Running Tasks Concurrently)*. This notebook tours the rest of the `asyncio` toolbox you'll use in real programs: queues, synchronisation primitives, async generators, bridging to blocking code, network streams and timeouts.

**In this notebook you will learn:**

- producer/consumer with `asyncio.Queue` (≈ `Channel<T>`)
- `asyncio.Lock`, `Event` and `Semaphore` (≈ `SemaphoreSlim.WaitAsync`)
- async generators and async comprehensions (≈ `IAsyncEnumerable<T>` / `await foreach`)
- running blocking code with `asyncio.to_thread` and `loop.run_in_executor`
- TCP clients and servers with asyncio streams
- timeouts with `asyncio.timeout` and protecting work with `asyncio.shield`
- how to choose between threads, processes and asyncio

## .NET mental model

| .NET | Python `asyncio` |
|---|---|
| `Channel<T>` (`WriteAsync` / `ReadAllAsync`) | `asyncio.Queue` (`await put` / `await get`) |
| `SemaphoreSlim(1)` used as an async lock | `asyncio.Lock` |
| `SemaphoreSlim(n).WaitAsync()` | `asyncio.Semaphore(n)` with `async with` |
| `TaskCompletionSource` / async manual-reset event | `asyncio.Event` |
| `IAsyncEnumerable<T>` + `yield return` | async generator (`async def` + `yield`) |
| `await foreach` | `async for` |
| `Task.Run(() => Blocking())` | `await asyncio.to_thread(blocking)` |
| `TcpListener` / `TcpClient` + `NetworkStream` | `asyncio.start_server` / `asyncio.open_connection` |
| `CancellationTokenSource.CancelAfter` / `WaitAsync(timeout)` | `async with asyncio.timeout(sec):` |

One key difference to keep in mind: .NET `await` resumes on thread-pool threads, so async C# code can still run in parallel. An asyncio event loop runs **everything on one thread** — coroutines interleave only at `await` points. That makes many race conditions impossible, but means a single blocking call freezes everything.

In a notebook the event loop is already running, so we use top-level `await` directly in cells. In a script you'd wrap the code in `async def main()` and call `asyncio.run(main())`.

## Producer/consumer with `asyncio.Queue`

```csharp
// C#
var channel = Channel.CreateBounded<int>(3);
async Task Produce() { for (int i = 0; i < 6; i++) await channel.Writer.WriteAsync(i); channel.Writer.Complete(); }
async Task Consume() { await foreach (var x in channel.Reader.ReadAllAsync()) Console.WriteLine(x); }
```

`asyncio.Queue` has the same API as `queue.Queue` *(see Standard Library › Concurrency › 002 · Locks, Events and Queues)* but `put()`/`get()`/`join()` are **awaitable**. With `maxsize`, `await q.put(x)` suspends while the queue is full — back-pressure, like a bounded channel.

In [ ]:
import asyncio

async def producer(q, n):
    for i in range(n):
        await q.put(i)                  # suspends if the queue is full
    print("producer done")

async def consumer(name, q, results):
    while True:
        item = await q.get()
        await asyncio.sleep(0.01)       # simulated async work
        results.append((name, item * item))
        q.task_done()

q = asyncio.Queue(maxsize=3)
results = []
consumers = [asyncio.create_task(consumer(f"c{i}", q, results)) for i in range(2)]

await producer(q, 6)
await q.join()                          # wait until every item is task_done()
for c in consumers:
    c.cancel()                          # consumers loop forever → cancel them

print(sorted(value for _, value in results))

Cancelling idle consumers after `q.join()` is the classic idiom. Since Python 3.13 you can instead **shut down** the queue — the equivalent of `channel.Writer.Complete()`: remaining items are still delivered, then `get()` raises `asyncio.QueueShutDown`.

In [ ]:
async def consume_until_shutdown(q, out):
    while True:
        try:
            item = await q.get()
        except asyncio.QueueShutDown:
            return
        out.append(item.upper())

q = asyncio.Queue()
out = []
async with asyncio.TaskGroup() as tg:
    tg.create_task(consume_until_shutdown(q, out))
    for word in ["red", "green", "blue"]:
        await q.put(word)
    q.shutdown()                        # like Writer.Complete()

print(out)

## `asyncio.Lock` — protecting state across an `await`

Because the loop runs one coroutine at a time, `counter += 1` *without* an `await` in between is safe in asyncio. The danger is a **check-then-act that spans an `await`**: another coroutine can run in the gap. Here two callers both see an empty cache and both "fetch":

In [ ]:
cache = {}
fetch_count = 0

async def get_config_unsafe():
    global fetch_count
    if "config" not in cache:
        fetch_count += 1
        await asyncio.sleep(0.05)       # the other coroutine runs here!
        cache["config"] = {"debug": False}
    return cache["config"]

await asyncio.gather(get_config_unsafe(), get_config_unsafe())
print("fetches without lock:", fetch_count)    # 2 — duplicated work

```csharp
// C# — SemaphoreSlim is the usual async lock
private readonly SemaphoreSlim _gate = new(1, 1);
await _gate.WaitAsync();
try { /* ... */ } finally { _gate.Release(); }
```

`asyncio.Lock` is used with `async with`, which awaits the acquire and always releases:

In [ ]:
cache = {}
fetch_count = 0
lock = asyncio.Lock()

async def get_config():
    global fetch_count
    async with lock:                    # C#: await _gate.WaitAsync(); try { ... } finally { Release(); }
        if "config" not in cache:
            fetch_count += 1
            await asyncio.sleep(0.05)
            cache["config"] = {"debug": False}
    return cache["config"]

await asyncio.gather(get_config(), get_config())
print("fetches with lock:", fetch_count)       # 1

⚠️ **Gotcha:** asyncio primitives (`asyncio.Lock`, `Event`, `Semaphore`, `Queue`) are **not thread-safe** and only work inside the event loop; `threading` primitives block the whole loop if you wait on them in a coroutine. Don't mix the two families.

## `asyncio.Event` and `asyncio.Semaphore`

`asyncio.Event` works like `threading.Event`, but `wait()` is awaited. `asyncio.Semaphore(n)` limits how many coroutines are inside a block at once — the standard way to cap concurrent HTTP requests, exactly like `SemaphoreSlim.WaitAsync()` in C#.

In [ ]:
ready = asyncio.Event()

async def waiter(name, log):
    await ready.wait()
    log.append(name)

log = []
tasks = [asyncio.create_task(waiter(n, log)) for n in ["alice", "bob"]]
await asyncio.sleep(0.02)
print("before set:", log)
ready.set()
await asyncio.gather(*tasks)
print("after set: ", sorted(log))

In [ ]:
limit = asyncio.Semaphore(2)
active = peak = 0

async def download(item):
    global active, peak
    async with limit:                   # C#: await sem.WaitAsync(); try { ... } finally { sem.Release(); }
        active += 1
        peak = max(peak, active)
        await asyncio.sleep(0.05)
        active -= 1
        return f"{item} ok"

print(await asyncio.gather(*(download(f"file{i}") for i in range(5))))
print("max concurrent downloads:", peak)       # 2

## Async generators and async comprehensions

```csharp
// C#
async IAsyncEnumerable<int> ReadingsAsync(int count)
{
    for (int i = 0; i < count; i++)
    {
        await Task.Delay(10);
        yield return i * 10;
    }
}
await foreach (var r in ReadingsAsync(3)) Console.WriteLine(r);
```

An `async def` function that contains `yield` is an **async generator** — Python's `IAsyncEnumerable<T>`. Consume it with `async for`:

In [ ]:
async def sensor_readings(count):
    for i in range(count):
        await asyncio.sleep(0.01)       # e.g. wait for the next sample
        yield i * 10

async for reading in sensor_readings(3):
    print("reading:", reading)

**Async comprehensions** use `async for` inside a comprehension (and you can `await` inside one too). They are like `await stream.ToListAsync()` with `System.Linq.Async`:

In [ ]:
readings = [r async for r in sensor_readings(5)]
big = [r async for r in sensor_readings(5) if r >= 20]
print(readings)
print(big)

async def double(x):
    await asyncio.sleep(0)
    return x * 2

print([await double(x) for x in range(3)])     # await inside a comprehension (runs sequentially)

⚠️ **Gotcha:** if you `break` out of an `async for`, the async generator is left suspended; its `finally` blocks run only when it is closed or garbage-collected. When the generator holds a resource, wrap it in `contextlib.aclosing(...)` so it is closed deterministically — like `await using` on an `IAsyncEnumerator`.

In [ ]:
from contextlib import aclosing

async def numbers():
    try:
        for i in range(100):
            yield i
    finally:
        print("generator closed")

async with aclosing(numbers()) as gen:
    async for n in gen:
        if n == 2:
            break
print("after loop")

## Running blocking code: `to_thread` and `run_in_executor`

A blocking call (`time.sleep`, a synchronous database driver, `requests.get`, heavy CPU work) inside a coroutine **freezes the whole event loop** — no other task can run. Offload it:

- `await asyncio.to_thread(fn, *args)` — runs `fn` in the default thread pool (≈ `await Task.Run(...)`)
- `await loop.run_in_executor(executor, fn, *args)` — the lower-level form; lets you pick the executor (e.g. a `ProcessPoolExecutor` for CPU-bound work, whose functions must be importable — see *Standard Library › Concurrency › 003 · Executors with concurrent.futures*)

In [ ]:
import time

def blocking_io(name):
    time.sleep(0.1)                     # a synchronous library call
    return f"{name} done"

start = time.perf_counter()
results = await asyncio.gather(
    asyncio.to_thread(blocking_io, "a"),
    asyncio.to_thread(blocking_io, "b"),
    asyncio.to_thread(blocking_io, "c"),
)
print(results, f"in ~{time.perf_counter() - start:.1f}s")   # ~0.1s, not 0.3s

In [ ]:
from concurrent.futures import ThreadPoolExecutor

loop = asyncio.get_running_loop()
with ThreadPoolExecutor(max_workers=2) as pool:
    r1 = loop.run_in_executor(pool, blocking_io, "x")
    r2 = loop.run_in_executor(pool, blocking_io, "y")
    print(await asyncio.gather(r1, r2))

**Tip:** going the other way — calling a coroutine from a *different* thread — uses `asyncio.run_coroutine_threadsafe(coro, loop)`, which returns a `concurrent.futures.Future`.

## TCP with asyncio streams

```csharp
// C#
var listener = new TcpListener(IPAddress.Loopback, 0);
listener.Start();
using var client = await listener.AcceptTcpClientAsync();
var stream = client.GetStream();   // then ReadAsync / WriteAsync
```

`asyncio.start_server(handler, host, port)` calls `handler(reader, writer)` for every connection, as a new task. `asyncio.open_connection(host, port)` returns a `(StreamReader, StreamWriter)` pair for the client side. Port `0` asks the OS for any free port.

In [ ]:
async def handle_echo(reader, writer):
    while line := await reader.readline():      # b"" at end of stream
        writer.write(line.upper())
        await writer.drain()                    # flow control: wait if the buffer is full
    writer.close()
    await writer.wait_closed()

server = await asyncio.start_server(handle_echo, "127.0.0.1", 0)
host, port = server.sockets[0].getsockname()[:2]
print("listening on", host, "port assigned:", port > 0)

In [ ]:
async def echo_client(*messages):
    reader, writer = await asyncio.open_connection("127.0.0.1", port)
    replies = []
    for msg in messages:
        writer.write(f"{msg}\n".encode())
        await writer.drain()
        replies.append((await reader.readline()).decode().rstrip())
    writer.close()
    await writer.wait_closed()
    return replies

print(await echo_client("hello", "asyncio streams"))

# several clients at once — the server handles each connection in its own task
print(await asyncio.gather(echo_client("alice"), echo_client("bob"), echo_client("carol")))

In [ ]:
server.close()
await server.wait_closed()
print("server closed")

**Tip:** `reader.readline()`, `reader.readexactly(n)` and `reader.readuntil(b"\r\n")` solve the "message framing" problem for you; lower-level sockets are covered in *Standard Library › Networking › 003 · Sockets*.

## Timeouts with `asyncio.timeout`

```csharp
// C#
using var cts = new CancellationTokenSource(TimeSpan.FromMilliseconds(50));
await SlowOperationAsync(cts.Token);       // throws OperationCanceledException
```

`async with asyncio.timeout(seconds):` (3.11+) cancels the code inside the block when the deadline passes and raises the built-in `TimeoutError`. Unlike .NET, you don't need to thread a token through every call — cancellation is injected at the current `await`.

In [ ]:
async def slow_operation():
    await asyncio.sleep(0.3)
    return "finished"

try:
    async with asyncio.timeout(0.05):
        await slow_operation()
except TimeoutError:
    print("TimeoutError: gave up after 0.05s")

async with asyncio.timeout(0.5):
    print(await slow_operation())      # fast enough

`asyncio.wait_for(coro, timeout)` is the older one-call form and is still fine. `asyncio.timeout(None)` means "no deadline", and the context manager's `reschedule()` can move the deadline.

⚠️ **Gotcha:** never catch and swallow `asyncio.CancelledError` inside the block — timeouts are implemented through cancellation, and suppressing it breaks them. If you must clean up, use `try/finally` or re-raise.

## Protecting work with `asyncio.shield`

When a timeout or cancellation hits an `await`, the awaited task is cancelled too. `asyncio.shield(task)` lets the *caller* stop waiting while the task itself **keeps running** — useful for work that must not be interrupted halfway (e.g. a commit).

The closest .NET analogue is `task.WaitAsync(timeout)`: it stops waiting without cancelling the underlying task (because .NET cancellation is cooperative, a task you stop awaiting just keeps running).

In [ ]:
saved = []

async def save_order():
    await asyncio.sleep(0.1)
    saved.append("order-42")
    return "saved"

task = asyncio.create_task(save_order())
try:
    async with asyncio.timeout(0.02):
        await asyncio.shield(task)      # the timeout cancels the shield, not the task
except TimeoutError:
    print("caller timed out; task cancelled?", task.cancelled())

print(await task, saved)                # the task finished anyway

⚠️ **Gotcha:** keep a reference to the shielded task (as above). The event loop holds only weak references to tasks, so a fire-and-forget task with no other reference can be garbage-collected before it finishes.

## Choosing: threads, processes or asyncio?

| Situation | Best fit | Why |
|---|---|---|
| Many concurrent network calls (HTTP, DB, sockets) with async libraries | **asyncio** | thousands of tasks on one thread, cheap |
| I/O with blocking / synchronous libraries | **threads** (`ThreadPoolExecutor`, `to_thread`) | GIL released while waiting |
| CPU-bound pure Python | **processes** (`ProcessPoolExecutor`, `multiprocessing`) | one GIL per process |
| CPU-bound in NumPy/hashlib/zlib etc. | threads often work | C code releases the GIL |
| CPU-bound, 3.14 free-threaded or sub-interpreters | `3.14t` threads / `InterpreterPoolExecutor` | true parallel threads |
| Async app that must call blocking/CPU code | asyncio + `to_thread` / `run_in_executor` | keeps the loop responsive |

In .NET you'd almost always answer "async/await plus the thread pool" — in Python the GIL makes the workload type (I/O vs CPU) the deciding factor *(see Standard Library › Concurrency › 001 · Threads)*.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Async producer/consumer | `Channel<T>` | `asyncio.Queue` |
| Complete the channel | `Writer.Complete()` | `q.shutdown()` (3.13+) |
| Async lock | `SemaphoreSlim(1)` + `WaitAsync` | `async with asyncio.Lock():` |
| Limit concurrency | `SemaphoreSlim(n).WaitAsync()` | `async with asyncio.Semaphore(n):` |
| Async signal | `TaskCompletionSource` | `asyncio.Event` |
| Async stream | `IAsyncEnumerable<T>` + `yield return` | `async def` + `yield` |
| Consume async stream | `await foreach` | `async for` / `[x async for x in g]` |
| Offload blocking work | `await Task.Run(...)` | `await asyncio.to_thread(fn)` |
| TCP server / client | `TcpListener` / `TcpClient` | `asyncio.start_server` / `open_connection` |
| Timeout | `CancelAfter` / `WaitAsync(timeout)` | `async with asyncio.timeout(s):` |
| Stop waiting, keep running | `task.WaitAsync(timeout)` | `asyncio.shield(task)` |

## Exercises

1. Write an async generator `countdown(n)` that yields `n, n-1, ..., 1` with `await asyncio.sleep(0.01)` between values, and collect it with an async comprehension.
2. Use an `asyncio.Semaphore(3)` to run 10 fake `fetch(i)` coroutines (each sleeps 0.02s and returns `i * i`) with at most 3 at a time. Print the results (in order) using `gather`.
3. Write `async def first_or_default(coro, timeout, default)` that returns the coroutine's result, or `default` if it doesn't finish within `timeout` seconds. Test it with a coroutine that sleeps 0.2s, using timeouts 0.05 and 0.5.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import asyncio

# 1 — async generator + async comprehension
async def countdown(n):
    for i in range(n, 0, -1):
        await asyncio.sleep(0.01)
        yield i

print([x async for x in countdown(5)])

# 2 — semaphore-limited gather (gather keeps argument order)
sem = asyncio.Semaphore(3)
async def fetch(i):
    async with sem:
        await asyncio.sleep(0.02)
        return i * i

print(await asyncio.gather(*(fetch(i) for i in range(10))))

# 3 — timeout with a default
async def first_or_default(coro, timeout, default):
    try:
        async with asyncio.timeout(timeout):
            return await coro
    except TimeoutError:
        return default

async def slow():
    await asyncio.sleep(0.2)
    return "value"

print(await first_or_default(slow(), 0.05, "default"))   # default
print(await first_or_default(slow(), 0.5, "default"))    # value
```

</details>